# Clasificador de frutas y verduras (20 clases)

Notebook completo del proyecto: descarga del dataset, EDA, entrenamiento de
dos modelos baseline (**MobileNetV2** y **EfficientNetV2B0**), comparacion,
exportacion a TFLite y verificacion de paridad con la app Android.

**Crucial:** el modelo se entrena para recibir imagenes **`float32` en
`[0.0, 1.0]`** (formato `channels-last`, `RGB`, `224x224`). Esto coincide
exactamente con lo que la app Android (`FruitImageClassifier.kt`) envia al
TFLite (`red/255f`, `green/255f`, `blue/255f`).


In [ ]:
# === 0. Dependencias ========================================================
# tf.keras.applications (MobileNetV2 y EfficientNetV2B0) viene incluido en
# TensorFlow >= 2.16, no necesitamos keras-hub.
!pip install -q "tensorflow>=2.16" "keras>=3.3" scikit-learn matplotlib seaborn prettytable Pillow imagehash

import os, sys, json, shutil, zipfile, random, time
from pathlib import Path
from collections import Counter

import numpy as np
import tensorflow as tf
import keras
from google.colab import drive

print(f"TF {tf.__version__}  |  Keras {keras.__version__}  |  Python {sys.version.split()[0]}")
print(f"GPUs disponibles: {tf.config.list_physical_devices('GPU')}")

# Reproducibilidad razonable (no determinista al 100% por GPU, pero estable).
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)


In [ ]:
# === 1. Montar Google Drive y crear estructura persistente ==================
drive.mount('/content/drive')

BASE_DIR        = Path('/content/drive/MyDrive/Proyecto_ML_Frutas_Verduras')
RAW_DIR         = BASE_DIR / 'data' / 'raw'           # ZIP original de Kaggle
PROCESSED_DIR   = BASE_DIR / 'data' / 'processed'     # Dataset DESCOMPRIMIDO (persistente)
ARTIFACTS_DIR   = BASE_DIR / 'artifacts'              # Modelos .keras / .tflite, metricas, plots
CHECKPOINT_DIR  = BASE_DIR / 'checkpoints'            # Pesos intermedios por epoca

for d in (RAW_DIR, PROCESSED_DIR, ARTIFACTS_DIR, CHECKPOINT_DIR):
    d.mkdir(parents=True, exist_ok=True)

DATASET_DRIVE_DIR = PROCESSED_DIR / 'Fruits_Vegetables_Dataset'  # raiz que vamos a usar
ZIP_NAME          = 'fruits-and-vegetables-dataset.zip'
ZIP_DRIVE_PATH    = RAW_DIR / ZIP_NAME

print(f"BASE_DIR          = {BASE_DIR}")
print(f"RAW_DIR           = {RAW_DIR}")
print(f"PROCESSED_DIR     = {PROCESSED_DIR}")
print(f"DATASET_DRIVE_DIR = {DATASET_DRIVE_DIR}")
print(f"ARTIFACTS_DIR     = {ARTIFACTS_DIR}")
print(f"CHECKPOINT_DIR    = {CHECKPOINT_DIR}")


In [ ]:
# === 2. Descargar el dataset de Kaggle (idempotente, token moderno) =========
# Kaggle ya permite autenticar con KAGGLE_API_TOKEN o ~/.kaggle/access_token.
# No se usa kaggle.json ni variables legacy KAGGLE_USERNAME / KAGGLE_KEY.

%pip install -q --upgrade kaggle

import os
from pathlib import Path

DATASET_REF = 'muhriddinmuxiddinov/fruits-and-vegetables-dataset'
KAGGLE_API_TOKEN_INLINE = 'KGAT_022e0ee7aeb8ac8eb3f5c79210e23029'

token = KAGGLE_API_TOKEN_INLINE.strip()
if not token:
    raise ValueError('No se ingreso ningun token de Kaggle.')
if not token.startswith('KGAT_'):
    print('Advertencia: el token no parece tener el formato actual de Kaggle (KGAT_...).')

KAGGLE_DIR = Path.home() / '.kaggle'
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)

ACCESS_TOKEN_DRIVE = BASE_DIR / 'kaggle_access_token'
ACCESS_TOKEN_RUNTIME = KAGGLE_DIR / 'access_token'

# El token debe existir antes de importar KaggleApi porque el paquete puede
# intentar autenticarse durante el import en Colab.
os.environ['KAGGLE_API_TOKEN'] = token
ACCESS_TOKEN_DRIVE.write_text(token, encoding='utf-8')
ACCESS_TOKEN_RUNTIME.write_text(token, encoding='utf-8')
try:
    ACCESS_TOKEN_RUNTIME.chmod(0o600)
except OSError:
    pass

print(f'Token de Kaggle guardado en Drive: {ACCESS_TOKEN_DRIVE}')

from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()
print('Autenticacion con Kaggle correcta.')

RAW_DIR.mkdir(parents=True, exist_ok=True)

# Descargamos el ZIP a Drive (no a /content) si todavia no esta.
if ZIP_DRIVE_PATH.exists():
    size_mb = ZIP_DRIVE_PATH.stat().st_size / (1024 * 1024)
    print(f'ZIP ya en Drive: {ZIP_DRIVE_PATH} ({size_mb:.1f} MB).')
else:
    print('Descargando dataset desde Kaggle...')
    zips_antes = set(RAW_DIR.glob('*.zip'))

    api.dataset_download_files(
        dataset=DATASET_REF,
        path=str(RAW_DIR),
        unzip=False,
        quiet=False,
    )

    if not ZIP_DRIVE_PATH.exists():
        zips_nuevos = [p for p in RAW_DIR.glob('*.zip') if p not in zips_antes]
        if len(zips_nuevos) == 1:
            zips_nuevos[0].replace(ZIP_DRIVE_PATH)

    if not ZIP_DRIVE_PATH.exists():
        raise FileNotFoundError(f'La descarga fallo, no se encontro {ZIP_DRIVE_PATH}')

    size_mb = ZIP_DRIVE_PATH.stat().st_size / (1024 * 1024)
    print(f'OK: {ZIP_DRIVE_PATH} ({size_mb:.1f} MB)')


In [ ]:
# === 3. Extraer el ZIP DIRECTAMENTE a Drive (idempotente) ===================
# El dataset queda persistido en Drive (`data/processed/Fruits_Vegetables_Dataset`).
# En corridas futuras saltamos esta celda automaticamente. Asi NO saturamos el
# disco /content (que se borra cuando el runtime de Colab se reinicia).

EXPECTED_SUBDIRS = ('Fruits', 'Vegetables')

def _dataset_ok(root: Path) -> bool:
    if not root.exists():
        return False
    for sub in EXPECTED_SUBDIRS:
        if not (root / sub).is_dir():
            return False
        classes = [d for d in (root / sub).iterdir() if d.is_dir()]
        if len(classes) != 10:
            return False
    return True

if _dataset_ok(DATASET_DRIVE_DIR):
    print(f"Dataset ya descomprimido en Drive: {DATASET_DRIVE_DIR}")
else:
    print("Descomprimiendo ZIP -> Drive (esto puede tardar 5-10 min por la velocidad de Drive)...")
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_DRIVE_PATH, 'r') as zf:
        zf.extractall(PROCESSED_DIR)

    # Algunos ZIPs traen una capa extra (`Fruits_Vegetables_Dataset/Fruits_Vegetables_Dataset/...`).
    candidatos = list(PROCESSED_DIR.rglob('Fruits'))
    if not candidatos:
        raise FileNotFoundError("No se encontro la carpeta 'Fruits' tras descomprimir.")
    raiz_real = candidatos[0].parent
    if raiz_real != DATASET_DRIVE_DIR:
        print(f"Moviendo dataset a la ubicacion canonica: {raiz_real} -> {DATASET_DRIVE_DIR}")
        if DATASET_DRIVE_DIR.exists():
            shutil.rmtree(DATASET_DRIVE_DIR)
        shutil.move(str(raiz_real), str(DATASET_DRIVE_DIR))

    assert _dataset_ok(DATASET_DRIVE_DIR), "El dataset no quedo bien instalado en Drive."
    print(f"Listo: {DATASET_DRIVE_DIR}")

# DATA_ROOT siempre apunta a la copia en Drive
DATA_ROOT = DATASET_DRIVE_DIR
for sub in EXPECTED_SUBDIRS:
    n_classes = sum(1 for _ in (DATA_ROOT / sub).iterdir() if _.is_dir())
    print(f"  {sub:12s} -> {n_classes} clases")


In [ ]:
# === 4. EDA: estructura, distribucion de clases, dimensiones, modos de color
# Esta seccion responde:
#   - cuantas imagenes por clase (balanceo)
#   - que dimensiones tienen (¿todas iguales? ¿hay outliers?)
#   - en que modo de color estan (RGB? con alpha? grises?)
#   - hay imagenes corruptas?
#   - hay duplicados perceptuales entre clases?
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

CATEGORIES = {
    'fruits':     DATA_ROOT / 'Fruits',
    'vegetables': DATA_ROOT / 'Vegetables',
}
EXPECTED_CLASSES = {
    'fruits': {'FreshApple','FreshBanana','FreshMango','FreshOrange','FreshStrawberry',
               'RottenApple','RottenBanana','RottenMango','RottenOrange','RottenStrawberry'},
    'vegetables': {'FreshBellpepper','FreshCarrot','FreshCucumber','FreshPotato','FreshTomato',
                   'RottenBellpepper','RottenCarrot','RottenCucumber','RottenPotato','RottenTomato'},
}

# 1. Validar estructura
for cat, root in CATEGORIES.items():
    encontradas = {p.name for p in root.iterdir() if p.is_dir()}
    faltan = EXPECTED_CLASSES[cat] - encontradas
    sobran = encontradas - EXPECTED_CLASSES[cat]
    if faltan or sobran:
        raise ValueError(f"[{cat}] faltan={faltan} | sobran={sobran}")
print("Estructura validada: 10 clases por categoria.")

# 2. Recolectar paths
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
registros = []  # filas para un DataFrame
for cat, root in CATEGORIES.items():
    for class_dir in sorted(root.iterdir()):
        if not class_dir.is_dir():
            continue
        for f in class_dir.iterdir():
            if f.is_file() and f.suffix.lower() in EXTS:
                registros.append({
                    'categoria': cat,
                    'clase':     class_dir.name,
                    'label':     f"{cat}__{class_dir.name}",
                    'path':      str(f),
                })

df = pd.DataFrame(registros)
print(f"Total imagenes: {len(df):,}")
print(df.groupby('categoria').size())

# 3. Distribucion por clase (barplot)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (cat, sub) in zip(axes, df.groupby('categoria')):
    conteo = sub['clase'].value_counts().sort_values(ascending=False)
    color = '#60a5fa' if cat == 'fruits' else '#34d399'
    ax.bar(range(len(conteo)), conteo.values, color=color)
    ax.set_xticks(range(len(conteo))); ax.set_xticklabels(conteo.index, rotation=70, ha='right', fontsize=8)
    ax.set_title(f"{cat.capitalize()} - imagenes por clase (n={int(conteo.sum()):,})")
    ax.set_ylabel('Imagenes')
plt.tight_layout(); plt.show()

# 4. Muestrear dimensiones y modo de color (sample para ser rapidos)
print("\nAnalizando dimensiones/modo de color (muestra de 1500 imgs)...")
muestra = df.sample(n=min(1500, len(df)), random_state=SEED)
filas = []
for r in muestra.itertuples(index=False):
    try:
        with Image.open(r.path) as im:
            filas.append({'w': im.width, 'h': im.height, 'mode': im.mode, 'label': r.label})
    except Exception:
        filas.append({'w': None, 'h': None, 'mode': 'corrupt', 'label': r.label})
dims = pd.DataFrame(filas).dropna()
print("Modos de color encontrados:")
print(dims['mode'].value_counts())
print(f"\nAncho:  min={dims['w'].min()}  max={dims['w'].max()}  media={dims['w'].mean():.0f}")
print(f"Alto:   min={dims['h'].min()}  max={dims['h'].max()}  media={dims['h'].mean():.0f}")
print(f"Aspect: min={(dims['w']/dims['h']).min():.2f}  max={(dims['w']/dims['h']).max():.2f}  "
      f"media={(dims['w']/dims['h']).mean():.2f}")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].hist(dims['w'], bins=40, color='#3b82f6', alpha=0.8); ax[0].set_title('Distribucion de ancho (px)')
ax[1].hist(dims['h'], bins=40, color='#10b981', alpha=0.8); ax[1].set_title('Distribucion de alto (px)')
plt.tight_layout(); plt.show()

# 5. Muestras visuales (1 imagen por clase, por categoria)
import matplotlib.image as mpimg
for cat, sub in df.groupby('categoria'):
    clases = sorted(sub['clase'].unique())
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for ax_i, clase in zip(axes.flatten(), clases):
        p = sub[sub['clase'] == clase].sample(1, random_state=SEED).iloc[0]['path']
        try:
            ax_i.imshow(mpimg.imread(p))
        except Exception:
            ax_i.text(0.5, 0.5, 'no img', ha='center', va='center')
        ax_i.set_title(clase, fontsize=9); ax_i.axis('off')
    fig.suptitle(f"Muestras - {cat}", fontweight='bold')
    plt.tight_layout(); plt.show()


In [ ]:
# === 5. Integridad: detectar imagenes corruptas o no-RGB convertibles ========
# El backbone espera 3 canales. Convertimos en disco las que no sean RGB para
# evitar sorpresas en tf.io.decode_image.
from PIL import Image, UnidentifiedImageError

corruptas, convertidas = [], 0
print(f"Revisando {len(df):,} imagenes...")

for p in df['path']:
    try:
        with Image.open(p) as im:
            im.verify()  # detecta archivos corruptos
        # Reabrir para inspeccionar modo (verify cierra el handle)
        with Image.open(p) as im:
            if im.mode not in ('RGB',):
                im.convert('RGB').save(p)
                convertidas += 1
    except (UnidentifiedImageError, OSError, ValueError):
        corruptas.append(p)

print(f"Imagenes corruptas: {len(corruptas)}  |  convertidas a RGB: {convertidas}")
for p in corruptas:
    try: os.remove(p)
    except OSError: pass

# Refrescar df despues de eliminar corruptas
df = df[~df['path'].isin(corruptas)].reset_index(drop=True)
print(f"Imagenes validas restantes: {len(df):,}")


## Fase 2 - Modelado y experimentacion

Comparamos dos arquitecturas de transfer learning sobre **un unico dataset
unificado de 20 clases** (10 frutas + 10 verduras) usando IMG=224, BATCH=32:

| Modelo | Params | Rol |
|---|---|---|
| **MobileNetV2** (ImageNet) | ~3.5M | Baseline mobile, exporta a `.tflite` chico, latencia baja. |
| **EfficientNetV2B0** (ImageNet) | ~7.1M | Backbone moderno, mejor accuracy en datasets medianos. |

**Receta de entrenamiento (identica para ambos):**

1. Split estratificado `70/15/15` (train/val/test).
2. Pipeline `tf.data` que entrega **`float32` en `[0.0, 1.0]`** (igual que el app Android).
3. Augmentacion dentro del modelo: flip, rotacion 30°, zoom 25%, shift 25%, contraste 20%, brillo 20% (todos en `[0,1]`).
4. **Fase 1 - cabeza (backbone congelado):** hasta 20 epocas, `Adam(lr=1e-3)`, EarlyStopping(patience=5).
5. **Fase 2 - fine-tune:** descongela ultimas 40 capas, hasta 15 epocas, `Adam(lr=1e-5)`.
6. `class_weight='balanced'`, `label_smoothing=0.05`, `ReduceLROnPlateau(factor=0.5, patience=2)`.

> **Importante**: la **unica** diferencia entre los dos modelos es cuanto
> escala/normaliza la imagen **dentro del modelo** (capa `Rescaling`), no en
> el pipeline. El input expuesto del modelo siempre es `float32 [0,1]`.


In [ ]:
# === 6. Split estratificado 70/15/15 sobre las 20 clases unificadas =========
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

filepaths  = df['path'].to_numpy()
labels_str = df['label'].to_numpy()

classes = sorted(set(labels_str.tolist()))
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
labels = np.array([class_to_idx[s] for s in labels_str])

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    filepaths, labels, test_size=0.30, stratify=labels, random_state=SEED)
X_va, X_te, y_va, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

weights_arr = compute_class_weight('balanced', classes=np.arange(len(classes)), y=y_tr)
class_weights = {i: float(w) for i, w in enumerate(weights_arr)}

split = {
    'classes': classes, 'class_to_idx': class_to_idx, 'idx_to_class': idx_to_class,
    'num_classes': len(classes),
    'X_train': X_tr, 'y_train': y_tr,
    'X_val':   X_va, 'y_val':   y_va,
    'X_test':  X_te, 'y_test':  y_te,
    'class_weights': class_weights,
}

print(f"== DATASET UNIFICADO ({split['num_classes']} clases) ==")
print(f"  total: {len(X_tr)+len(X_va)+len(X_te):,}  | train={len(X_tr):,}  val={len(X_va):,}  test={len(X_te):,}")
print("  distribucion en train:")
counts = Counter(y_tr.tolist())
for idx in range(split['num_classes']):
    print(f"    {idx:2d} -> {idx_to_class[idx]:32s} {counts[idx]:>5,}  (w={class_weights[idx]:.3f})")


In [ ]:
# === 7. Persistir labels.txt / class_names.json ==============================
# IMPORTANTISIMO: el orden de las clases en labels.txt define el orden de las
# salidas del modelo. Esta lista DEBE coincidir con app/src/main/assets/labels.txt
# de la app Android. El orden alfabetico que usa sklearn coincide con el que
# espera la app (fruits__... seguido de vegetables__...).
class_names = [split['idx_to_class'][i] for i in range(split['num_classes'])]

labels_path      = ARTIFACTS_DIR / 'labels.txt'
class_names_path = ARTIFACTS_DIR / 'class_names.json'
class_index_path = ARTIFACTS_DIR / 'class_index.json'

labels_path.write_text('\n'.join(class_names) + '\n', encoding='utf-8')
class_names_path.write_text(json.dumps(class_names, ensure_ascii=False, indent=2), encoding='utf-8')
class_index_path.write_text(
    json.dumps({str(i): n for i, n in enumerate(class_names)}, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print(f"labels.txt       -> {labels_path}")
print(f"class_names.json -> {class_names_path}")
print(f"class_index.json -> {class_index_path}")
for i, n in enumerate(class_names):
    print(f"  {i:2d} -> {n}")


In [ ]:
# === 8. Pipeline tf.data: float32 RGB [0,1], 224x224 ========================
# Este pipeline produce EXACTAMENTE el mismo tensor que la app Android arma
# desde el Bitmap (ver FruitImageClassifier.kt#toModelInputBuffer):
#
#     red/255f, green/255f, blue/255f  ->  float32 channels-last
#
# Asi la entrada del modelo expuesto siempre es [0,1] y la inferencia en
# Android se comporta igual que la inferencia en Colab.
IMG_SIZE   = 224
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

def _load_image(path, label):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method='bilinear')
    img = tf.cast(img, tf.float32) / 255.0    # <<< [0,1], identico a la app
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img, label

def make_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(min(len(paths), 10_000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(_load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    ds = ds.prefetch(AUTOTUNE)
    return ds

pipelines = {
    'train': make_dataset(split['X_train'], split['y_train'], training=True),
    'val':   make_dataset(split['X_val'],   split['y_val'],   training=False),
    'test':  make_dataset(split['X_test'],  split['y_test'],  training=False),
}
print(f"IMG_SIZE={IMG_SIZE}  BATCH_SIZE={BATCH_SIZE}")
print(f"train batches~{(len(split['X_train']) + BATCH_SIZE - 1) // BATCH_SIZE}")

# Augmentacion: trabaja en [0,1]. Solo afecta cuando training=True.
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal',           seed=SEED),
    tf.keras.layers.RandomRotation(30 / 360,           seed=SEED),
    tf.keras.layers.RandomZoom(0.25,                   seed=SEED),
    tf.keras.layers.RandomTranslation(0.25, 0.25,      seed=SEED),
    tf.keras.layers.RandomContrast(0.20,               seed=SEED),
    tf.keras.layers.RandomBrightness(0.20, value_range=(0.0, 1.0), seed=SEED),  # <- [0,1]
], name='data_augmentation')
print("Augmentacion configurada (rango [0,1]).")

# Smoke test del pipeline: confirmar rango y shape
for x, y in pipelines['train'].take(1):
    print(f"\nSmoke test pipeline:  x.shape={tuple(x.shape)}  x.dtype={x.dtype}  "
          f"x.min={float(tf.reduce_min(x)):.4f}  x.max={float(tf.reduce_max(x)):.4f}")
    assert x.dtype == tf.float32
    assert float(tf.reduce_min(x)) >= 0.0 and float(tf.reduce_max(x)) <= 1.0,         "El pipeline debe entregar valores en [0,1]."
print("Pipeline OK: rango [0,1], dtype float32, RGB channels-last.")


In [ ]:
# === 9. Builders: MobileNetV2 y EfficientNetV2B0 ============================
# El input expuesto del modelo es [0,1] float32.  Cada arquitectura aplica
# INTERNAMENTE su propia normalizacion via Rescaling para igualar el rango
# que esperaba su pre-entrenamiento (sin necesidad de tocar la app):
#
#   * MobileNetV2  espera [-1, 1]   ->  Rescaling(scale=2.0, offset=-1.0)
#   * EfficientNetV2B0 trae una capa Normalization INTERNA que asume rango
#     [0, 255]  ->  Rescaling(scale=255.0, offset=0.0) antes del backbone
#
# Con esto NO importa que el app divida por 255: el modelo se encarga de
# remapear al rango que cada backbone necesita.
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV2, EfficientNetV2B0


def _build_classifier(backbone, rescale_layer, num_classes, name):
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image_0_1')
    x = data_augmentation(inputs)            # no-op fuera de training
    x = rescale_layer(x)                     # [0,1] -> rango nativo del backbone
    x = backbone(x, training=False)          # BN siempre en eval mode
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(0.30, name='dropout_head')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    return models.Model(inputs, outputs, name=name)


def build_mobilenetv2(num_classes, name='mobilenetv2'):
    base = MobileNetV2(include_top=False, weights='imagenet',
                       input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    rescale = layers.Rescaling(scale=2.0, offset=-1.0, name='to_minus1_plus1')
    return _build_classifier(base, rescale, num_classes, name), base


def build_efficientnetv2b0(num_classes, name='efficientnetv2b0'):
    base = EfficientNetV2B0(include_top=False, weights='imagenet',
                            input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    # EfficientNetV2B0 (tf.keras.applications) hace su propia Normalization
    # INTERNA, asumiendo input en [0,255]. Por eso multiplicamos por 255.
    rescale = layers.Rescaling(scale=255.0, name='to_0_255')
    return _build_classifier(base, rescale, num_classes, name), base


def train_model(name, build_fn, num_classes, class_weights, train_ds, val_ds,
                epochs_head=20, epochs_finetune=15,
                lr_head=1e-3, lr_finetune=1e-5,
                unfreeze_from=-40, label_smoothing=0.05):
    print("\n" + "=" * 72)
    print(f"Entrenando: {name}")
    print("=" * 72)

    model, base = build_fn(num_classes, name=name)
    print(f"  params totales:    {model.count_params():,}")
    print(f"  params entrenables (head): {sum(int(np.prod(v.shape)) for v in model.trainable_weights):,}")

    ckpt_best = CHECKPOINT_DIR / f'{name}_best.keras'
    ckpt_last = CHECKPOINT_DIR / f'{name}_last.keras'

    cb_head = [
        callbacks.ModelCheckpoint(str(ckpt_best), save_best_only=True,
                                   monitor='val_accuracy', mode='max', verbose=1),
        callbacks.ModelCheckpoint(str(ckpt_last), save_best_only=False, verbose=0),
        callbacks.EarlyStopping(monitor='val_accuracy', patience=5,
                                 mode='max', restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2,
                                     min_lr=1e-6, verbose=1),
    ]
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()  # label_smoothing solo en categorical
    top3    = tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3')

    # ---- Fase 1: cabeza ----------------------------------------------------
    model.compile(optimizer=optimizers.Adam(lr_head),
                  loss=loss_fn, metrics=['accuracy', top3])
    t0 = time.time()
    h_head = model.fit(train_ds, validation_data=val_ds,
                       epochs=epochs_head, callbacks=cb_head,
                       class_weight=class_weights, verbose=2)
    t_head = time.time() - t0

    # ---- Fase 2: fine-tune --------------------------------------------------
    base.trainable = True
    if unfreeze_from < 0:
        for layer in base.layers[:unfreeze_from]:
            layer.trainable = False
    print(f"  fine-tune: {sum(1 for l in base.layers if l.trainable)} / {len(base.layers)} layers entrenables")

    cb_ft = [
        callbacks.ModelCheckpoint(str(ckpt_best), save_best_only=True,
                                   monitor='val_accuracy', mode='max', verbose=1),
        callbacks.ModelCheckpoint(str(ckpt_last), save_best_only=False, verbose=0),
        callbacks.EarlyStopping(monitor='val_accuracy', patience=5,
                                 mode='max', restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2,
                                     min_lr=1e-7, verbose=1),
    ]
    model.compile(optimizer=optimizers.Adam(lr_finetune),
                  loss=loss_fn, metrics=['accuracy', top3])
    t0 = time.time()
    h_ft = model.fit(train_ds, validation_data=val_ds,
                     epochs=epochs_finetune, callbacks=cb_ft,
                     class_weight=class_weights, verbose=2)
    t_ft = time.time() - t0

    hist = {
        'head':     {k: list(v) for k, v in h_head.history.items()},
        'finetune': {k: list(v) for k, v in h_ft.history.items()},
        'time_head_sec':  t_head,
        'time_ft_sec':    t_ft,
        'time_total_sec': t_head + t_ft,
    }
    print(f"\n  best checkpoint -> {ckpt_best}  ({(t_head + t_ft) / 60:.1f} min total)")
    return model, hist, str(ckpt_best)


In [ ]:
# === 10. Entrenar AMBOS modelos y comparar ==================================
# Tiempo aproximado en T4 (BATCH=32, IMG=224, ~8.4k imgs train):
#   MobileNetV2      : 12-18 min
#   EfficientNetV2B0 : 22-32 min
#   TOTAL aprox.     : 35-50 min
modelos_a_entrenar = [
    ('mobilenetv2',      build_mobilenetv2),
    ('efficientnetv2b0', build_efficientnetv2b0),
]
resultados = {}
for nombre, build_fn in modelos_a_entrenar:
    model, hist, ckpt = train_model(
        name=nombre, build_fn=build_fn,
        num_classes=split['num_classes'],
        class_weights=split['class_weights'],
        train_ds=pipelines['train'], val_ds=pipelines['val'],
        epochs_head=20, epochs_finetune=15,
        lr_head=1e-3, lr_finetune=1e-5, unfreeze_from=-40,
    )
    resultados[nombre] = {'model': model, 'hist': hist, 'ckpt': ckpt}

print("\n" + "=" * 72)
print("Resumen de entrenamiento")
print("=" * 72)
for nombre, r in resultados.items():
    val_acc = r['hist']['finetune'].get('val_accuracy', r['hist']['head'].get('val_accuracy', [0]))[-1]
    print(f"  {nombre:20s} best={r['ckpt']}  val_acc(final)={val_acc:.4f}  "
          f"time={r['hist']['time_total_sec']/60:.1f} min")


In [ ]:
# === 10b. Convertir .keras a TFLite para Android (reanudable desde Drive) ===
# Si el runtime se desconecta, esta celda reconstruye `resultados` desde
# CHECKPOINT_DIR y ARTIFACTS_DIR sin reentrenar.

import json


def _mejor_val_accuracy(hist):
    vals = list(hist.get('head', {}).get('val_accuracy', [])) + list(hist.get('finetune', {}).get('val_accuracy', []))
    return max(vals) if vals else 0.0


def _model_names_esperados():
    if 'modelos_a_entrenar' in globals() and modelos_a_entrenar:
        return [nombre for nombre, _ in modelos_a_entrenar]
    return ['mobilenetv2', 'efficientnetv2b0']


def _load_historial_desde_drive():
    hist_path = ARTIFACTS_DIR / 'training_history.json'
    if not hist_path.exists():
        return {}
    try:
        return json.loads(hist_path.read_text(encoding='utf-8'))
    except Exception as exc:
        print(f'Advertencia: no se pudo leer {hist_path}: {exc}')
        return {}


def _ensure_resultados_from_drive():
    global resultados

    if 'resultados' in globals() and isinstance(resultados, dict) and len(resultados) > 0:
        print('`resultados` ya existe en memoria. Se usaran esos checkpoints.')
        return

    historial = _load_historial_desde_drive()
    nombres = _model_names_esperados()

    resultados = {}
    faltantes = []

    for nombre in nombres:
        ckpt = CHECKPOINT_DIR / f'{nombre}_best.keras'
        if not ckpt.exists():
            faltantes.append(str(ckpt))
            continue

        model = tf.keras.models.load_model(ckpt, compile=False)
        hist = historial.get(nombre, {'head': {}, 'finetune': {}, 'time_head_sec': 0.0, 'time_ft_sec': 0.0, 'time_total_sec': 0.0})

        resultados[nombre] = {
            'model': model,
            'hist': hist,
            'ckpt': str(ckpt),
        }

    if faltantes:
        raise FileNotFoundError(
            'Faltan checkpoints en Drive. Ejecuta entrenamiento o verifica CHECKPOINT_DIR:\n- '
            + '\n- '.join(faltantes)
        )

    print('`resultados` reconstruido desde Drive:')
    for nombre, r in resultados.items():
        print(f"  {nombre:20s} ckpt={r['ckpt']}")


def convertir_checkpoint_a_tflite(model_name, checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'No existe el checkpoint: {checkpoint_path}')

    keras_path = ARTIFACTS_DIR / f'{model_name}.keras'
    tflite_path = ARTIFACTS_DIR / f'{model_name}.tflite'

    shutil.copy2(checkpoint_path, keras_path)
    print(f'keras  -> {keras_path} ({keras_path.stat().st_size / (1024 * 1024):.2f} MB)')

    model = tf.keras.models.load_model(keras_path, compile=False)
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = []  # Float32: la app envia float32 [0,1].
    tflite_bytes = converter.convert()
    tflite_path.write_bytes(tflite_bytes)
    print(f'tflite -> {tflite_path} ({tflite_path.stat().st_size / (1024 * 1024):.2f} MB)')

    return {'keras': str(keras_path), 'tflite': str(tflite_path)}


ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
_ensure_resultados_from_drive()

artefactos = {}
for nombre, r in resultados.items():
    print('\n' + '=' * 72)
    print(f'Convirtiendo {nombre}')
    print('=' * 72)
    artefactos[nombre] = convertir_checkpoint_a_tflite(nombre, r['ckpt'])

# Elegimos ganador por validacion si hay historial; si no, tomamos el primero.
best_name = max(resultados.keys(), key=lambda n: _mejor_val_accuracy(resultados[n]['hist']))
app_tflite_src = Path(artefactos[best_name]['tflite'])
app_tflite_dst = ARTIFACTS_DIR / 'fruit_classifier.tflite'
shutil.copy2(app_tflite_src, app_tflite_dst)

print('\n' + '=' * 72)
print(f'Modelo elegido por validacion: {best_name}')
print(f'MODELO PARA LA APP: {app_tflite_dst}')
print('  copialo a: app/src/main/assets/fruit_classifier.tflite')
print(f'  labels:    {ARTIFACTS_DIR / "labels.txt"}  ->  app/src/main/assets/labels.txt')
print('=' * 72)


In [ ]:
# === 11. Curvas de entrenamiento =============================================
import matplotlib.pyplot as plt

def _concat(hist, key):
    return list(hist['head'].get(key, [])) + list(hist['finetune'].get(key, []))

fig, axes = plt.subplots(len(resultados), 2, figsize=(14, 4.5 * len(resultados)))
if len(resultados) == 1:
    axes = axes.reshape(1, 2)

for row, (nombre, r) in enumerate(resultados.items()):
    hist = r['hist']
    train_acc = _concat(hist, 'accuracy'); val_acc = _concat(hist, 'val_accuracy')
    train_loss = _concat(hist, 'loss');    val_loss = _concat(hist, 'val_loss')
    ft_start = len(hist['head'].get('accuracy', []))
    xs = list(range(1, len(train_acc) + 1))

    a = axes[row, 0]
    a.plot(xs, train_acc, '-o', label='train', color='#3498db', markersize=4)
    a.plot(xs, val_acc,   '-o', label='val',   color='#e74c3c', markersize=4)
    if 0 < ft_start < len(xs):
        a.axvline(ft_start + 0.5, color='gray', linestyle='--', alpha=0.6, label='inicio fine-tune')
    a.set_title(f'{nombre} - Accuracy', fontweight='bold')
    a.set_xlabel('Epoch'); a.set_ylabel('Accuracy'); a.grid(alpha=0.3); a.legend()

    b = axes[row, 1]
    b.plot(xs, train_loss, '-o', label='train', color='#3498db', markersize=4)
    b.plot(xs, val_loss,   '-o', label='val',   color='#e74c3c', markersize=4)
    if 0 < ft_start < len(xs):
        b.axvline(ft_start + 0.5, color='gray', linestyle='--', alpha=0.6, label='inicio fine-tune')
    b.set_title(f'{nombre} - Loss', fontweight='bold')
    b.set_xlabel('Epoch'); b.set_ylabel('Loss'); b.grid(alpha=0.3); b.legend()

plt.tight_layout()
plot_path = ARTIFACTS_DIR / 'training_curves.png'
plt.savefig(plot_path, dpi=120, bbox_inches='tight'); plt.show()
print(f"Curvas guardadas en: {plot_path}")

# Persistir historial en JSON
hist_json = {
    nombre: {
        'head':           {k: [float(v) for v in vs] for k, vs in r['hist']['head'].items()},
        'finetune':       {k: [float(v) for v in vs] for k, vs in r['hist']['finetune'].items()},
        'time_head_sec':  float(r['hist']['time_head_sec']),
        'time_ft_sec':    float(r['hist']['time_ft_sec']),
        'time_total_sec': float(r['hist']['time_total_sec']),
    }
    for nombre, r in resultados.items()
}
(ARTIFACTS_DIR / 'training_history.json').write_text(
    json.dumps(hist_json, ensure_ascii=False, indent=2), encoding='utf-8')
print(f"Historial guardado en: {ARTIFACTS_DIR / 'training_history.json'}")


In [ ]:
# === 12. Evaluacion en TEST + comparacion + matrices de confusion ===========
import seaborn as sns
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from prettytable import PrettyTable

target_names = [split['idx_to_class'][i] for i in range(split['num_classes'])]
y_true = np.concatenate([y for _, y in pipelines['test']], axis=0)

resumen_modelos = {}
for nombre, r in resultados.items():
    model = r['model']
    loss, acc, top3 = model.evaluate(pipelines['test'], verbose=0)
    y_prob = model.predict(pipelines['test'], verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    report = classification_report(y_true, y_pred, target_names=target_names,
                                   digits=3, zero_division=0, output_dict=True)
    cm = confusion_matrix(y_true, y_pred)
    resumen_modelos[nombre] = {
        'loss': float(loss), 'accuracy': float(acc), 'top3': float(top3),
        'macro_f1':    float(report['macro avg']['f1-score']),
        'weighted_f1': float(report['weighted avg']['f1-score']),
        'training_time_min': float(r['hist']['time_total_sec'] / 60),
        'y_pred': y_pred, 'y_prob': y_prob,
        'report_dict': report, 'confusion_matrix': cm,
    }
    print(f">>> {nombre}: acc={acc:.4f} top3={top3:.4f} macro_f1={resumen_modelos[nombre]['macro_f1']:.4f}")
    (ARTIFACTS_DIR / f'classification_report_{nombre}.json').write_text(
        json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

# Tabla comparativa
pt = PrettyTable(field_names=['Modelo','Accuracy','Top-3','Macro F1','Loss','Tiempo (min)'])
for nombre, m in resumen_modelos.items():
    pt.add_row([nombre, f"{m['accuracy']:.4f}", f"{m['top3']:.4f}",
                f"{m['macro_f1']:.4f}", f"{m['loss']:.4f}",
                f"{m['training_time_min']:.1f}"])
print("\n" + str(pt))

# Eleccion del ganador (max accuracy, desempate por macro_f1)
best_name = max(resumen_modelos.keys(),
                key=lambda n: (resumen_modelos[n]['accuracy'], resumen_modelos[n]['macro_f1']))
print(f"\nGANADOR: {best_name}")

# Persistir resumen comparativo
metrics_summary = {
    'best_model': best_name,
    'models': {n: {k: v for k, v in m.items() if k not in ('y_pred', 'y_prob', 'report_dict', 'confusion_matrix')}
               for n, m in resumen_modelos.items()},
}
(ARTIFACTS_DIR / 'metrics_summary.json').write_text(
    json.dumps(metrics_summary, ensure_ascii=False, indent=2), encoding='utf-8')

# Matriz de confusion del ganador
cm = resumen_modelos[best_name]['confusion_matrix']
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
plt.figure(figsize=(14, 12))
sns.heatmap(cm_norm, annot=False, cmap='Blues', xticklabels=target_names, yticklabels=target_names,
            cbar_kws={'label': 'Recall por fila'})
plt.title(f'Matriz de confusion (normalizada) - {best_name}', fontweight='bold')
plt.xlabel('Predicho'); plt.ylabel('Real'); plt.xticks(rotation=70, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8); plt.tight_layout()
cm_path = ARTIFACTS_DIR / f'confusion_matrix_{best_name}.png'
plt.savefig(cm_path, dpi=120, bbox_inches='tight'); plt.show()
print(f"Matriz de confusion guardada en: {cm_path}")


In [ ]:
# === 13. Exportar artefactos + COPIA LISTA PARA LA APP ANDROID ==============
# Para cada baseline generamos EXACTAMENTE 2 archivos:
#   <name>.keras    (modelo completo Keras, para reanudar o evaluar)
#   <name>.tflite   (TFLite float32, lo que va a Android)
#
# Para el GANADOR ademas copiamos:
#   fruit_classifier.tflite   (mismo bytes que <best>.tflite, nombre que espera la app)
def export_artifacts(model_name, checkpoint_path):
    checkpoint_path = Path(checkpoint_path)

    keras_path = ARTIFACTS_DIR / f'{model_name}.keras'
    shutil.copy2(checkpoint_path, keras_path)
    print(f"  keras  -> {keras_path.name:42s} ({keras_path.stat().st_size/(1024*1024):.2f} MB)")

    model = tf.keras.models.load_model(keras_path, compile=False)
    tflite_bytes = tf.lite.TFLiteConverter.from_keras_model(model).convert()
    tflite_path = ARTIFACTS_DIR / f'{model_name}.tflite'
    tflite_path.write_bytes(tflite_bytes)
    print(f"  tflite -> {tflite_path.name:42s} ({tflite_path.stat().st_size/(1024*1024):.2f} MB)")

    return {'keras': str(keras_path), 'tflite': str(tflite_path)}


artefactos = {}
for nombre, r in resultados.items():
    print(f"\n== Export {nombre.upper()} ==")
    artefactos[nombre] = export_artifacts(nombre, r['ckpt'])

# Copia LISTA PARA LA APP del modelo ganador (float32, igual que <best>.tflite)
app_tflite_src = Path(artefactos[best_name]['tflite'])
app_tflite_dst = ARTIFACTS_DIR / 'fruit_classifier.tflite'
shutil.copy2(app_tflite_src, app_tflite_dst)
print(f"\nMODELO PARA LA APP: {app_tflite_dst}")
print(f"  copialo a: app/src/main/assets/fruit_classifier.tflite")
print(f"  labels:    {ARTIFACTS_DIR / 'labels.txt'}  ->  app/src/main/assets/labels.txt")


In [ ]:
# === 14. requirements.txt consolidado =======================================
req = (BASE_DIR / 'requirements.txt')
contenido = (
    'tensorflow>=2.16\n'
    'keras>=3.3\n'
    'numpy\n'
    'pandas\n'
    'matplotlib\n'
    'seaborn\n'
    'scikit-learn\n'
    'prettytable\n'
    'Pillow\n'
    'imagehash\n'
    'kaggle\n'
    'gradio\n'
    'streamlit\n'
)
req.write_text(contenido, encoding='utf-8')
print(f'requirements.txt -> {req}')
print(contenido)


In [ ]:
# === 15. Sanity check de PARIDAD con la app Android =========================
# Esta celda emula PASO A PASO lo que hace FruitImageClassifier.kt:
#   1. Decodifica una imagen como RGB con PIL (igual que ImageDecoder).
#   2. La redimensiona a 224x224 con bilinear (igual que createScaledBitmap).
#   3. Extrae R/G/B por separado y divide cada canal por 255 -> float32 [0,1].
#   4. Arma el tensor (1, 224, 224, 3) en row-major (igual que el ByteBuffer).
#   5. Corre la inferencia con el .tflite EXPORTADO.
#
# Si las predicciones top-3 aqui coinciden con las del modelo .keras de
# referencia (que ya validamos contra el test set), entonces la app Android
# va a producir EXACTAMENTE los mismos resultados al cargar el archivo.
from PIL import Image

def android_like_preprocess(img_path, size=224):
    """Reproduce el flujo del .kt: Bitmap -> ARGB -> (R,G,B)/255 float32."""
    with Image.open(img_path) as im:
        im = im.convert('RGB').resize((size, size), Image.BILINEAR)
        arr = np.asarray(im, dtype=np.uint8)         # (H, W, 3) RGB uint8
    arr = arr.astype(np.float32) / 255.0             # [0,1]
    return np.expand_dims(arr, 0)                    # (1, 224, 224, 3)

# Cargamos el .tflite que se va a copiar a la app
interp = tf.lite.Interpreter(model_path=str(ARTIFACTS_DIR / 'fruit_classifier.tflite'))
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print(f"TFLite input : shape={inp['shape']}  dtype={inp['dtype'].__name__}")
print(f"TFLite output: shape={out['shape']}  dtype={out['dtype'].__name__}")
assert tuple(inp['shape']) == (1, IMG_SIZE, IMG_SIZE, 3), "El TFLite debe aceptar [1,224,224,3]"
assert inp['dtype'] == np.float32, "El TFLite debe aceptar float32 (la app envia float32)."

# Tomamos 1 imagen aleatoria por clase del TEST SET y vemos las predicciones
print("\n" + "=" * 72)
print("Predicciones reproduciendo el preprocess de la app Android (TFLite f32)")
print("=" * 72)
class_names = [split['idx_to_class'][i] for i in range(split['num_classes'])]
aciertos = 0; total = 0
for cls_idx in range(split['num_classes']):
    mask = split['y_test'] == cls_idx
    if not mask.any():
        continue
    path = split['X_test'][np.where(mask)[0][0]]
    x = android_like_preprocess(path, IMG_SIZE)
    interp.set_tensor(inp['index'], x)
    interp.invoke()
    probs = interp.get_tensor(out['index'])[0]
    top3_idx = np.argsort(probs)[::-1][:3]
    top3 = ', '.join(f"{class_names[int(i)]}={probs[int(i)]:.3f}" for i in top3_idx)
    ok = int(top3_idx[0]) == cls_idx
    aciertos += int(ok); total += 1
    flag = 'OK' if ok else 'X '
    print(f"  [{flag}] real={class_names[cls_idx]:32s}  top1={class_names[int(top3_idx[0])]:32s}  | {top3}")

print(f"\nTop-1 accuracy reproduciendo Android preprocess: {aciertos}/{total} = {aciertos/total:.2%}")
print("\nSi ves accuracy alto aqui, la app Android va a clasificar bien tras copiar:")
print(f"  {ARTIFACTS_DIR / 'fruit_classifier.tflite'}  ->  app/src/main/assets/fruit_classifier.tflite")
print(f"  {ARTIFACTS_DIR / 'labels.txt'}              ->  app/src/main/assets/labels.txt")


In [ ]:
# === 16. Inventario final de artefactos =====================================
print(f"\n=== Mapa final de artefactos en Drive ({ARTIFACTS_DIR}) ===")
print(f"  {'archivo':60s} {'tamano':>10s}")
print(f"  {'-'*60} {'-'*10}")
for p in sorted(ARTIFACTS_DIR.iterdir()):
    if p.is_file():
        mb = p.stat().st_size / (1024 * 1024)
        print(f"  {p.name:60s} {mb:>7.2f} MB")

print("\n=== Para la APP Android ===")
print(f"  1) Copia    {ARTIFACTS_DIR / 'fruit_classifier.tflite'}")
print(f"     a       app/src/main/assets/fruit_classifier.tflite")
print(f"  2) Copia    {ARTIFACTS_DIR / 'labels.txt'}")
print(f"     a       app/src/main/assets/labels.txt")
print(f"  3) Recompila la app. NO hace falta cambiar el codigo Kotlin: el")
print(f"     modelo ya acepta el tensor [1,224,224,3] float32 en [0,1] que")
print(f"     la app actual envia (red/255f, green/255f, blue/255f).")
